# Mixed detection (roadmap step 7)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohaaa825/Skolfie/blob/main/notebooks/mixed.ipynb)

Tests whether the detector can tell **Mixed** texts (part human, part AI) apart from purely
human or purely AI ones, and how well it finds the AI parts:

* builds test texts by splicing human and AI passages from the same source and topic
  (human-then-AI, AI-then-human, and sandwiches), plus pure human and pure AI texts,
* scores every text and every chunk of a few sentences with your trained model,
* tunes the Mixed rule on validation texts and reports on test texts, next to a simpler rule
  that only looks at the whole-text score.

**Use a GPU runtime** (T4): the model computes GPT-2 features for every chunk. Then
**Runtime → Run all**. It should take about 10 to 20 minutes.

Writes `My Drive/skolfie/reports/mixed-<model>/`.

## 1. Settings

In [ ]:
DATA_DIR = "/content/drive/MyDrive/skolfie/data/processed"
REPORTS_DIR = "/content/drive/MyDrive/skolfie/reports"
MODELS_DIR = "/content/drive/MyDrive/skolfie/models"

REPO = "mohaaa825/Skolfie"
BRANCH = "main"

MODEL = "transformer"  # a file in MODELS_DIR: "transformer", "features" or "stacked"
N_MIXED = 2000         # spliced texts per set (validation and test)
N_PURE = 1000          # pure human and pure AI texts per set, each
CHUNK_WORDS = 60       # chunk size in words
MAX_FALSE_MIXED = 0.05 # at most this share of pure texts may be called Mixed

## 2. Check the GPU, mount Drive, clone and install

In [ ]:
import os
import subprocess

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU; scoring the chunks with GPT-2 features will be very slow.")

from google.colab import drive

drive.mount("/content/drive")
assert os.path.exists(f"{MODELS_DIR}/{MODEL}.joblib"), f"No saved model {MODEL}.joblib yet."


def secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.isdir("/content/Skolfie/.git"):
    subprocess.run(["git", "-C", "/content/Skolfie", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/Skolfie"],
                   check=True)

%cd /content/Skolfie
!git log --oneline -1
!pip install -q -e ".[features]"

## 3. Build the Mixed sets, score and report

In [ ]:
import sys

hf_token = secret("HF_TOKEN")
env = dict(os.environ, **({"HF_TOKEN": hf_token} if hf_token else {}))
cmd = ["python", "-u", "scripts/eval_mixed.py",
       "--corpus", f"{DATA_DIR}/corpus.parquet",
       "--splits", f"{DATA_DIR}/splits.parquet",
       "--model", f"{MODELS_DIR}/{MODEL}.joblib",
       "--n-mixed", str(N_MIXED), "--n-pure", str(N_PURE),
       "--target-words", str(CHUNK_WORDS),
       "--max-false-mixed", str(MAX_FALSE_MIXED),
       "--out-dir", f"{REPORTS_DIR}/mixed-{MODEL}"]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                        bufsize=1, env=env)
for line in proc.stdout:
    sys.stdout.write(line)
if proc.wait() != 0:
    raise RuntimeError(f"eval_mixed.py failed with exit code {proc.returncode}")

Tell Claude in the project thread when this has finished; it reads the report from Drive
directly.